In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# step1_make_tsub.py
# -*- coding: utf-8 -*-
import json, os
import pandas as pd

CLO_JSONL = "/content/drive/MyDrive/train_clo_results.jsonl"     # 需含 set_id 與 CLO 欄位（total_clo/clo/CLO 任一）
MET_JSON  = "/content/drive/MyDrive/train_met.json"             # 需含 set_id / activity / met_value
OUT_PATH  = "/content/drive/MyDrive/train_temperature.json"

def _read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_clo_df(jsonl_path: str) -> pd.DataFrame:
    df = pd.DataFrame(_read_jsonl(jsonl_path))
    for k in ["total_clo", "clo", "CLO"]:
        if k in df.columns:
            df = df.rename(columns={k: "total_clo"})
            break
    if "set_id" not in df.columns or "total_clo" not in df.columns:
        raise ValueError("train_clo_results.jsonl 需要 set_id 與 CLO 欄位（total_clo / clo / CLO）") # Modified error message
    df["set_id"] = df["set_id"].astype(str)
    df["total_clo"] = df["total_clo"].astype(float)
    return df[["set_id","total_clo"]]

def load_met_df(json_path: str) -> pd.DataFrame:
    df = pd.read_json(json_path)
    need = {"set_id","activity","met_value"}
    if not need.issubset(df.columns):
        raise ValueError("train_met.json 需要欄位：set_id / activity / met_value") # Modified error message
    df["set_id"] = df["set_id"].astype(str)
    df["met_value"] = df["met_value"].astype(float)
    return df[["set_id","activity","met_value"]]

def calc_tsub(iclo: float, met: float) -> float:
    H = met * 58.2  # W/m^2
    tsub = 33.5 - 3.0*iclo - (0.08 + 0.05*iclo) * H
    return round(tsub, 1)

def main():
    clo_df = load_clo_df(CLO_JSONL)
    met_df = load_met_df(MET_JSON)
    df = clo_df.merge(met_df, on="set_id", how="inner")
    df["TSUB_target_C"] = df.apply(lambda r: calc_tsub(r["total_clo"], r["met_value"]), axis=1)
    # 以「清單(list of objects)」方式輸出
    records = df[["set_id","total_clo","activity","met_value","TSUB_target_C"]].to_dict(orient="records")
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"[OK] 已輸出：{OUT_PATH}（共 {len(records)} 筆）")

if __name__ == "__main__":
    main()

[OK] 已輸出：/content/drive/MyDrive/train_temperature.json（共 17011 筆）


In [ ]:
# -*- coding: utf-8 -*-
# step2_generate_titles.py

import json
import os
import math
import re
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from transformers import pipeline
import kagglehub
import glob

print("正在設定環境與路徑...")

# ==============================================================================
# Part 1: 路徑與設定 (請根據你的環境確認)
# ==============================================================================

# --- 輸入檔案 ---
# 步驟 1 產生的溫度資料
TEMP_JSON = "/content/drive/MyDrive/train_temperature.json" # Modified to use train_temperature.json

# --- 輸出檔案 ---
OUT_JSON = "/content/drive/MyDrive/new_polyvore_outfit_titles.json"

# --- Kaggle 資料集設定 ---
# 這會自動下載並設定 Polyvore 資料集的路徑
try:
    dnepozitek_polyvore_outfits_path = kagglehub.dataset_download('dnepozitek/polyvore-outfits')
    dataset_root = os.path.join(dnepozitek_polyvore_outfits_path, "polyvore_outfits")
    IMAGES_DIR = os.path.join(dataset_root, "images")
    SPLIT_DIR  = os.path.join(dataset_root, "disjoint")
    META_PATH  = os.path.join(dataset_root, "polyvore_item_metadata.json")
    print(f"Polyvore 資料集路徑: {dataset_root}")
except Exception as e:
    print(f"[錯誤] 無法下載或找到 Kaggle 資料集，請確認 API 金鑰與網路連線: {e}")
    exit() # 如果資料集無法載入，則無法繼續

# --- 模型設定 ---
# 使用你之前已設定好的本地 Gemma-3 模型
try:
    google_gemma_3_transformers_gemma_3_4b_it_1_path = kagglehub.model_download('google/gemma-3/Transformers/gemma-3-4b-it/1')
    print("Gemma-3 模型路徑設定完成。")
except Exception as e:
    print(f"[錯誤] 無法下載或找到 Gemma-3 模型: {e}")
    exit()

# --- 其他設定 ---
# 拼貼圖暫存目錄
COLLAGE_DIR = "collages_for_titles"
os.makedirs(COLLAGE_DIR, exist_ok=True)

正在設定環境與路徑...
Using Colab cache for faster access to the 'polyvore-outfits' dataset.
Polyvore 資料集路徑: /kaggle/input/polyvore-outfits/polyvore_outfits
Gemma-3 模型路徑設定完成。


In [ ]:
# ==============================================================================
# Part 2: 輔助函式
# ==============================================================================

print("正在載入輔助函式與 Metadata...")

# --- 載入單品 metadata ---
if os.path.exists(META_PATH):
    meta_df = pd.read_json(META_PATH, orient="index")
    meta_df.index = meta_df.index.astype(str)
    meta_df.index.name = "item_id"
    print(f"成功載入 {len(meta_df)} 筆單品 metadata。")
else:
    raise FileNotFoundError(f"找不到 metadata 檔案: {META_PATH}")

def image_path(item_id: str) -> str | None:
    """根據 item_id 取得圖片的絕對路徑"""
    p = os.path.join(IMAGES_DIR, f"{item_id}.jpg")
    return p if os.path.exists(p) else None

def make_collage(image_paths: list[str], out_path: str, tile=(224, 224), cols=4) -> str | None:
    """製作拼貼圖"""
    if not image_paths: return None
    w, h = tile
    rows = math.ceil(len(image_paths) / cols)
    canvas = Image.new("RGB", (cols * w, rows * h), (255, 255, 255))
    r = c = 0
    for p in image_paths:
        try:
            with Image.open(p) as im:
                im = im.convert("RGB").resize(tile, Image.LANCZOS)
                canvas.paste(im, (c * w, r * h))
            c += 1
            if c == cols:
                c = 0
                r += 1
        except Exception as e:
            # print(f"[警告] 讀取圖片失敗: {p}, {e}") # 可選：印出錯誤
            pass
    canvas.save(out_path)
    return out_path

def _parse_titles_json(js) -> dict[str, str]:
    """解析不同格式的 outfit title json"""
    m = {}
    if isinstance(js, dict):
        for k, v in js.items():
            title = ""
            if isinstance(v, str):
                title = v
            elif isinstance(v, dict):
                title = v.get("title") or v.get("url_name")
            if title:
                m[str(k)] = title.strip()
    return m

def load_outfit_titles_map(base_dir: str) -> dict[str, str]:
    """載入套裝原始標題"""
    candidates = [
        os.path.join(base_dir, "polyvore_outfit_titles.json"),
        os.path.join(base_dir, "nondisjoint", "polyvore_outfit_titles.json"),
    ]
    titles = {}
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    js = json.load(f)
                    titles.update(_parse_titles_json(js))
            except Exception as e:
                print(f"[警告] 載入標題檔失敗: {p}, {e}")
    print(f"成功載入 {len(titles)} 筆套裝原始標題。")
    return titles

def _load_train_records() -> list[dict]:
    """載入 train.json"""
    path = os.path.join(SPLIT_DIR, "train.json") # Modified to load train.json
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    raise FileNotFoundError(f"找不到 train.json 於 {path}") # Modified error message

def _extract_outfit(rec: dict) -> tuple[str, list[str]]:
    """從 record 中提取 set_id 和 item_ids"""
    set_id = str(rec.get("set_id", ""))
    items = rec.get("items", [])
    item_ids = [str(it.get("item_id", "")) for it in items]
    return set_id, [i for i in item_ids if i]


# ==============================================================================
# Part 3: Gemini 模型與 Prompt 設定
# ==============================================================================

def setup_pipeline():
    """設定 Gemma-3 推論 pipeline"""
    print("正在初始化 Gemma-3 pipeline (可能需要幾分鐘)...")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    pipe = pipeline(
        task="image-text-to-text",
        model=google_gemma_3_transformers_gemma_3_4b_it_1_path,
        device="cuda",
        dtype=torch.bfloat16,
        trust_remote_code=True
    )
    print("Pipeline 初始化完成。")
    return pipe

def create_generation_prompt(temp: float, clo: float, desc: str) -> str:
    """建立用於生成標題的 Prompt"""
    temp_int = int(round(temp, 0))
    prompt = fprompt = f"""
    You are a fashion expert who writes creative, natural, and search-friendly outfit titles.

    Analyze the provided outfit image and the following contextual data to generate a new title.
    The title should sound like a phrase a user might type when searching for this kind of outfit online.

    **Outfit Data:**
    - **Recommended Temperature:** {temp:.1f}°C
    - **Clothing Insulation:** {clo:.2f} CLO
    - **Original Title:** {desc}

    **Guidelines:**
    1. Begin the title with the temperature, like "{int(round(temp, 0))}°C, ".
    2. Combine weather feeling and the outfit’s vibe or implied scene based on its visual cues (style, material, layering, color, accessories) into one concise, fluent sentence.
    3. Reflect practical logic between temperature, CLO value, and the outfit — as if you understand why these clothes match the weather and activity.
    4. The tone should be aspirational yet realistic, suitable for fashion recommendation platforms.
    5. Avoid listing exact items (e.g., "shirt", "pants"); focus on the style or scene instead .
    6. Output **only** the final title string, without any explanations or extra text.

    **Your Thought Process (implicit, do not include in output):**
    - Think about the thermal comfort logic: since the temperature is {temp:.1f}°C, imagine what it would actually feel like.
    - Consider how the {clo:.2f} CLO value supports comfort in that condition.
    - Infer a plausible scene primarily from visual cues in the outfit.
    - Your goal is to make the title sound like something users would actually search for when dressing for similar weather and activities.

    **Output:** A single English title that follows the above rules.
    """
    return prompt.strip()

import re

def clean_model_output(raw_text: str, temp_text: str, delimiter: str = "° C") -> str | None:
    """
    保持以 '° C' 分割的邏輯不變：
    1) 把 '℃'、'°C'（無空格）、'°  C' 等全部正規化成 '° C'
    2) 用第一個 '° C' 當分隔，取後段描述
    3) 前綴用你計算的溫度 + '° C, ' 重組
    """
    if not raw_text:
        return None

    # 取第一行非空
    line = next((l.strip() for l in str(raw_text).splitlines() if l.strip()), "")
    if not line:
        return None

    # 先做正規化：把所有度數寫法統一成 '° C'
    norm = (line
            .replace("℃", "° C")   # 日/中常見
            .replace("°C", "° C")  # 無空格 -> 有空格
            .replace("°  C", "° C")  # 多空白 -> 一個空白
           )

    # 如果已經有 '° C'，沿用你現在的分割邏輯
    idx = norm.find(delimiter)
    if idx >= 0:
        desc = norm[idx + len(delimiter):].lstrip(" ,，")
        return f"{temp_text}{delimiter}, {desc}" if desc else None

    # 若還是找不到，處理像「-7°C Cozy...」這種：抓掉前綴溫度與可選逗號，取後段
    m = re.match(r'^\s*[+-]?\d+(?:\.\d+)?\s*°\s*C\s*[,，]?\s*(.*)$', line)
    if m:
        desc = m.group(1).strip()
        return f"{temp_text}{delimiter}, {desc}" if desc else None

    return None



# ==============================================================================
# Part 4: 主要執行流程
# ==============================================================================
def main():
    # --- 1. 載入所有需要的資料 ---
    print("\n--- 開始載入資料 ---")
    try:
        temp_df = pd.read_json(TEMP_JSON)
        print(f"成功載入 {len(temp_df)} 筆溫度資料。")
    except Exception as e:
        print(f"[錯誤] 無法讀取 {TEMP_JSON}，請先執行 step1_make_tsub.py。錯誤: {e}")
        return

    original_titles = load_outfit_titles_map(dataset_root)
    train_records = _load_train_records() # Modified to load train.json

    # 建立一個從 set_id 到 item_ids 的對照表，方便快速查找
    set_id_to_items = {
        sid: iids for sid, iids in (_extract_outfit(rec) for rec in train_records) if sid
    }
    print(f"已建立 {len(set_id_to_items)} 筆 set_id 對照表。")

    # --- 新增: 設定已存在的拼貼圖目錄 ---
    EXISTING_COLLAGE_DIR = "/content/drive/MyDrive/collages_train_demo" # Consider changing this to a train collage directory if needed
    print(f"將會優先從此路徑讀取已存在的拼貼圖: {EXISTING_COLLAGE_DIR}")
    os.makedirs(EXISTING_COLLAGE_DIR, exist_ok=True) # 確保目錄存在，避免錯誤

    # --- 2. 設定模型 ---
    pipe = setup_pipeline()

    # --- 3. 載入已完成的結果，以支援中斷續跑 ---
    if os.path.exists(OUT_JSON):
        try:
            with open(OUT_JSON, 'r', encoding='utf-8') as f:
                results = json.load(f)
            print(f"已載入 {len(results)} 筆先前已完成的結果。")
        except json.JSONDecodeError:
            print(f"[警告] 輸出檔案 {OUT_JSON} 格式錯誤，將建立新檔案。")
            results = {}
    else:
        results = {}

    # --- 4. 迭代處理每一筆資料 ---
    print("\n--- 開始生成標題 ---")

    # 只處理尚未完成的項目
    remaining_df = temp_df[~temp_df['set_id'].astype(str).isin(results.keys())]
    if remaining_df.empty:
        print("所有項目皆已完成！")
        return

    print(f"尚有 {len(remaining_df)} 筆資料待處理。")

    for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="生成套裝標題"):
        set_id = str(row['set_id'])
        collage_path = None  # 初始化拼貼圖路徑

        # A: 準備拼貼圖 (邏輯已修改)
        # 步驟 1: 優先檢查預存的拼貼圖路徑
        existing_path = os.path.join(EXISTING_COLLAGE_DIR, f"{set_id}.jpg") # Consider changing this to look in a train collage directory
        if os.path.exists(existing_path):
            collage_path = existing_path
        else:
            # 步驟 2: 如果預存的圖片不存在，則自動製作新的
            item_ids = set_id_to_items.get(set_id)
            if not item_ids:
                # print(f"[警告] 找不到 set_id {set_id} 的單品清單，無法製作拼貼圖，跳過。")
                continue

            paths = [p for p in (image_path(i) for i in item_ids) if p]
            # 將新製作的圖存到暫存區
            temp_collage_path = os.path.join(COLLAGE_DIR, f"{set_id}.jpg")
            collage_path = make_collage(paths, temp_collage_path)

        # 如果最終還是沒有有效的拼貼圖路徑 (例如製作失敗)，則跳過
        if not collage_path:
            # print(f"[警告] 無法為 set_id {set_id} 找到或建立拼貼圖，跳過。")
            continue

        # Display the collage image to be processed
        from IPython.display import display
        display(Image.open(collage_path))


        # B, D, E: 從 row 中取得資料
        total_clo = row['total_clo']
        activity = row['activity']
        tsub = row['TSUB_target_C']


        # C: 取得簡易描述
        simple_desc = original_titles.get(set_id, "None") # 若找不到原始標題，給一個預設值

        # 建立 Prompt
        user_prompt = create_generation_prompt(tsub, total_clo, simple_desc)

        # 準備給模型的訊息
        messages = [{"role": "user", "content": [
            {"type": "image", "image": collage_path},
            {"type": "text", "text": user_prompt}
        ]}]

        # 呼叫模型生成
        try:
            output = pipe(text=messages, max_new_tokens=100) # 限制生成長度
            raw_text = output[0]["generated_text"][-1]["content"]
            print(raw_text)
            temp_text = str(row['TSUB_target_C'])
            generated_title = clean_model_output(raw_text, temp_text)
        except Exception as e:
            print(f"\n[錯誤] 處理 set_id {set_id} 時模型推論失敗: {e}")
            generated_title = f"Error: Model inference failed."
            torch.cuda.empty_cache() # 清理 CUDA cache

        # 只在成功解析時才寫入檔案；否則跳過不寫
        if generated_title :
            results[set_id] = {"title": generated_title}
            print(f"\n[成功] set_id {set_id} 生成標題：{generated_title}")
            with open(OUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
        else:
            print(f"[略過] set_id {set_id} 未產生有效 title，不寫檔。")
            continue

    print(f"\n[成功] 標題生成完成！\n結果已儲存至: {OUT_JSON} (共 {len(results)} 筆)")


if __name__ == "__main__":
    main()

正在載入輔助函式與 Metadata...
成功載入 251008 筆單品 metadata。

--- 開始載入資料 ---
成功載入 17011 筆溫度資料。
成功載入 68306 筆套裝原始標題。
已建立 16995 筆 set_id 對照表。
將會優先從此路徑讀取已存在的拼貼圖: /content/drive/MyDrive/collages_train_demo
正在初始化 Gemma-3 pipeline (可能需要幾分鐘)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Pipeline 初始化完成。
已載入 35140 筆先前已完成的結果。

--- 開始生成標題 ---
所有項目皆已完成！
